In [ ]:
import matplotlib.pyplot as plt

# Read the data from the file
frames = []
rgy_values = []

with open("rgyr_vs_frame.txt", "r") as file:
    for line in file:
        frame, rg = line.split()
        frames.append(int(frame))
        rgy_values.append(float(rg))

# Get the initial radius of gyration value for percentage calculation
initial_rg = rgy_values[0]

# Calculate the percentage values
percentage_values = [(rg / initial_rg) * 100 for rg in rgy_values]

# Create the plot
fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot radius of gyration on the primary y-axis
ax1.plot(frames, rgy_values, marker='o', linestyle='-', color='b', label='Radius of Gyration', linewidth=1)
ax1.set_xlabel('Frame')
ax1.set_ylabel('Radius of Gyration (Å)', color='b')
ax1.tick_params(axis='y', labelcolor='b')
ax1.grid(True)

# Create a secondary y-axis for percentage values
ax2 = ax1.twinx()
ax2.plot(frames, percentage_values, marker='o', linestyle='--', color='r', label='Percentage of Initial Value', linewidth=1)
ax2.set_ylabel('Percentage of Initial Radius of Gyration (%)', color='r')
ax2.tick_params(axis='y', labelcolor='r')

# Add titles and labels
plt.title('Radius of Gyration vs. Frame')

# Save the plot as a PNG file
plt.savefig("radius_of_gyration_vs_frame_Nov13.png")

# Show the plot
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def load_xyz(filename):
    """Load coordinates from an .xyz file."""
    data = np.loadtxt(filename, skiprows=2, usecols=(1, 2, 3))
    return data

def compute_rdf(coords, r_max=100.0, dr=1.0, sphere_radius=2500.0):
    """
    Compute radial distribution function (RDF) for coordinates within a sphere.
    Uses voxelized neighbor search for efficiency.
    """
    N = coords.shape[0]
    rho = N / ((4/3) * np.pi * sphere_radius**3)

    # Bin edges and centers
    bins = np.arange(0, r_max + dr, dr)
    r_centers = 0.5 * (bins[:-1] + bins[1:])
    g_r = np.zeros_like(r_centers)

    # Build voxel grid
    cell_size = r_max
    mins = coords.min(axis=0)
    maxs = coords.max(axis=0)
    grid_shape = np.ceil((maxs - mins) / cell_size).astype(int)
    voxels = {}

    def voxel_index(pos):
        return tuple(((pos - mins) / cell_size).astype(int))

    for i, pos in enumerate(coords):
        voxels.setdefault(voxel_index(pos), []).append(i)

    # Neighbor search using voxel grid
    for voxel_key, indices in voxels.items():
        neighbor_voxels = [
            (voxel_key[0] + dx, voxel_key[1] + dy, voxel_key[2] + dz)
            for dx in (-1, 0, 1)
            for dy in (-1, 0, 1)
            for dz in (-1, 0, 1)
        ]
        for i in indices:
            for nv in neighbor_voxels:
                if nv not in voxels:
                    continue
                for j in voxels[nv]:
                    if j <= i:
                        continue
                    rij = np.linalg.norm(coords[i] - coords[j])
                    if rij < r_max:
                        bin_index = int(rij / dr)
                        g_r[bin_index] += 2  # count both i→j and j→i

    # Normalize
    shell_volumes = 4/3 * np.pi * ((bins[1:]**3) - (bins[:-1]**3))
    ideal_counts = rho * shell_volumes * N
    g_r /= ideal_counts

    return r_centers, g_r

In [ ]:
bin_file = "./Nov13_1/data/coords/dna_Nov13_1_1_preequil.bin"    # change this if needed
coords = read_bin(bin_file)

r, g_r = compute_rdf(coords, r_max=100.0, dr=1.0, sphere_radius=2000.0)

plt.figure(figsize=(6,4))
plt.plot(r, g_r, color="purple", lw=2)
plt.xlabel("r (Å)")
plt.ylabel("g(r) — normalized neighbor density")
plt.title("Radial Distribution Function of DNA Beads")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
bin_file = "./Nov13_1/data/coords/dna_Nov13_1_1_postequil.bin"    # change this if needed
coords = read_bin(bin_file)

r, g_r = compute_rdf(coords, r_max=100.0, dr=1.0, sphere_radius=2000.0)

plt.figure(figsize=(6,4))
plt.plot(r, g_r, color="purple", lw=2)
plt.xlabel("r (Å)")
plt.ylabel("g(r) — normalized neighbor density")
plt.title("Radial Distribution Function of DNA Beads")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import argparse
from scipy.optimize import curve_fit

def read_bin(path, order="row"):
    data = np.fromfile(path, dtype=np.float64)
    n_coords = data.size
    if n_coords % 3 != 0:
        raise ValueError("Binary file size is not divisible by 3")
    N = n_coords // 3
    if order == "row":
        coords = data.reshape((N, 3))
    elif order == "col":
        coords = np.empty((N, 3), dtype=np.float64)
        coords[:, 0] = data[0:N]
        coords[:, 1] = data[N:2*N]
        coords[:, 2] = data[2*N:3*N]
    else:
        raise ValueError("order must be 'row' or 'col'")
    return np.array(coords)


def read_xyz(filename):
    with open(filename) as f:
        lines = f.readlines()
    N = int(lines[0].strip())
    coords = []
    for line in lines[2:2+N]:
        parts = line.split()
        coords.append([float(parts[1]), float(parts[2]), float(parts[3])])
    return np.array(coords)

def exp_decay(x, lp, B):
    return np.exp(-x / lp) * np.cos(2*np.pi * x / B)

def bond_correlation(coords, max_lag=None):
    bonds = coords[1:] - coords[:-1]
    bond_lengths = np.linalg.norm(bonds, axis=1)
    bonds /= bond_lengths[:, None]   # normalize bond vectors
    l0 = bond_lengths.mean()
    print(f"l0 = {l0:.2f}")
    n_bonds = len(bonds)
    if max_lag is None:
        max_lag = n_bonds - 1

    corr = []
    distances = []
    for j in range(1, max_lag+1):
        dots = np.sum(bonds[:-j] * bonds[j:], axis=1)
        corr.append(dots.mean())
        distances.append(j * l0)
    return np.array(distances), np.array(corr), l0


# parser = argparse.ArgumentParser(description="Compute bond correlation function and estimate persistence length")
# parser.add_argument("xyz_file", help=".xyz file with bead coordinates")
# parser.add_argument("--maxlag", type=int, default=50, help="Maximum lag (number of bonds)")
# parser.add_argument("--output", default="persistence", help="Output prefix")
# args = parser.parse_args()

In [ ]:
xyz_file = "./Nov13_1/data/coords/dna_Nov13_1_1_postequil.bin" 
maxlag=50


coords = read_bin(xyz_file) / 10
distances, corr, l0 = bond_correlation(coords, max_lag=maxlag)

# fit exponential to correlation
try:
    popt, _ = curve_fit(exp_decay, distances, corr, p0=[45.0, 2000.0])
    lp_fit = popt[0]
    print(f"Estimated persistence length lp = {lp_fit:.2f} nm")
    B_fit = popt[1]
    print(f"Estimated confinement length scale B = {B_fit:.2f} nm")
except RuntimeError:
    lp_fit = None
    print("Fit failed, plotting raw data only.")

plt.figure()
plt.plot(distances, corr, 'o', label="data")
if lp_fit:
    plt.plot(distances, exp_decay(distances, lp_fit, B_fit), '-', label = fr"fit $e^{{-j \cdot l_0 / l_p}} \cos(2\pi j l_0/B)$, $l_p$={lp_fit:.1f}, B={B_fit:.1f}")
plt.xlabel("Contour distance (nm)")
plt.ylabel(r"Bond correlation ⟨$u_i \cdot  u_{i+j}$⟩")
plt.title("Bond Correlation vs Distance")
plt.legend()
plt.ylim(-0.2, 1.0)
plt.show()

In [ ]:
xyz_file = "./Nov13_1/data/coords/dna_Nov13_1_1_preequil.bin" 
maxlag=50


coords = read_bin(xyz_file) / 10
distances, corr, l0 = bond_correlation(coords, max_lag=maxlag)

# fit exponential to correlation
try:
    popt, _ = curve_fit(exp_decay, distances, corr, p0=[45.0, 2000.0])
    lp_fit = popt[0]
    print(f"Estimated persistence length lp = {lp_fit:.2f} nm")
    B_fit = popt[1]
    print(f"Estimated confinement length scale B = {B_fit:.2f} nm")
except RuntimeError:
    lp_fit = None
    print("Fit failed, plotting raw data only.")

plt.figure()
plt.plot(distances, corr, 'o', label="data")
if lp_fit:
    plt.plot(distances, exp_decay(distances, lp_fit, B_fit), '-', label = fr"fit $e^{{-j \cdot l_0 / l_p}} \cos(2\pi j l_0/B)$, $l_p$={lp_fit:.1f}, B={B_fit:.1f}")
plt.xlabel("Contour distance (nm)")
plt.ylabel(r"Bond correlation ⟨$u_i \cdot  u_{i+j}$⟩")
plt.title("Bond Correlation vs Distance")
plt.legend()
plt.ylim(-0.2, 1.0)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from pathlib import Path

mm = 1 / 25.4  # millimeters to inches conversion for figure sizing

def contact_mat_alignment(mat):
    """
    Aligns a symmetric contact matrix so that all diagonals collapse into a single mean row.
    """
    N = mat.shape[0]
    aligned_mat = np.zeros_like(mat)
    aligned_mat[0, :] = mat[0, :]

    for i in range(1, N):
        aligned_mat[i, :] = np.roll(mat[i, :], -i)
    
    return aligned_mat


def multi_fits(ax, locus_size, regimes, contact_sum):
    """
    Perform log–log fits in specified regimes and plot the power-law fits.
    """
    for regime in reversed(regimes):
        start, end = regime
        z = np.arange(start, end) * (locus_size / 1000.0)
        X = np.ones((len(z), 2), dtype=np.float64)
        X[:, 1] = np.log(z)
        y = np.log(contact_sum[start:end])

        # Linear least squares in log space
        beta = np.linalg.lstsq(X, y, rcond=None)[0]
        slope = beta[1]

        print(f"Fit {start}-{end} kbp: slope = {slope:.3f}")

        y_fit = np.exp(beta[0] + beta[1] * np.log(z))
        label = fr"{start}-{end} kbp: $s = -${-slope:.3f}"

        ax.plot(z, y_fit, ls="-", c="w", lw=2.0, alpha=0.65)
        ax.plot(z, y_fit, ls="--", lw=1.5, alpha=0.9, label=label)


def plot_contact_law(contact_map_file, output_label="contact_law", locus_size=1000):
    """
    Load a normalized contact map (.npy) and plot the contact probability vs genomic distance.
    """
    mat = np.load(contact_map_file)
    N = mat.shape[0]

    # Align and average diagonals
    aligned_mat = contact_mat_alignment(mat)
    contact_sum = np.mean(aligned_mat, axis=0)

    # Define plot settings
    lower_lim, upper_lim = 1, N//2#min(101, N)
    regimes = [[1, 11], [1, 51], [1, 101]]

    za = np.arange(lower_lim, upper_lim) * (locus_size / 1000.0)

    fig_size = [120*2, 40*2]
    fig, ax = plt.subplots(figsize=(fig_size[0] * mm, fig_size[1] * mm))

    ax.set_xlabel(r"Genomic Distance [kbp] - $x$", fontsize=8)
    ax.set_ylabel(r"Contact Probability - $P(x)$", fontsize=7)

    ax.plot(za, contact_sum[lower_lim:upper_lim],
            c='k', lw=1.0, alpha=0.35, marker='.', markersize=6,
            fillstyle='none', label="Data")

    multi_fits(ax, locus_size, regimes, contact_sum)

    ax.set_xlim(za[0], za[-1])
    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.legend(fontsize=7)
    ax.tick_params(axis='x', labelsize=7)
    ax.tick_params(axis='y', labelsize=7)
    plt.tight_layout()

    out_pdf = f"{output_label}_contact_law.pdf"
    out_png = f"{output_label}_contact_law.png"
    plt.savefig(out_pdf, dpi=300)
    plt.savefig(out_png, dpi=300)
    print(f"✅ Saved plots: {out_pdf}, {out_png}")

date = 'Oct30'
output_label = date
minute = '11'
CG = 1000
contact_map_file = "./results/"+date+"/contact_map_delta_cg"+str(CG)+date+"_"+minute+".npy"
plot_contact_law(contact_map_file, output_label)


In [ ]:
date = 'Oct29'
output_label = date
minute = '11'
CG = 1000
contact_map_file = "./results/"+date+"/contact_map_delta_cg"+str(CG)+date+"_"+minute+".npy"
plot_contact_law(contact_map_file, output_label)

In [ ]:
date = 'Nov04'
output_label = date
minute = '11'
CG = 1000
contact_map_file = "./results/"+date+"/contact_map_delta_cg"+str(CG)+date+"_"+minute+".npy"
plot_contact_law(contact_map_file, output_label)

In [ ]:
plot_contact_law("3C_contact_matrix.npy", output_label)

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
N = 544
locus_size = 1000
# Define plot settings
lower_lim, upper_lim = 1, N//2#min(101, N)
regimes = [[1, 11], [1, 51], [1, 101]]

za = np.arange(lower_lim, upper_lim) * (locus_size / 1000.0)

fig_size = [120*2, 40*2]
fig, ax = plt.subplots(figsize=(fig_size[0] * mm, fig_size[1] * mm))

ax.set_xlabel(r"Genomic Distance [kbp] - $x$", fontsize=12)
ax.set_ylabel(r"Contact Probability - $P(x)$", fontsize=12)

contact_map_files = [#"Oct30.npy",
                     "Nov18.npy",
                     "Oct29.npy",
                     "Nov04.npy",
                    "3C_contact_matrix_normalized.npy"]

colors = ['g', 'b', 'r', 'k']
labels = ['No blocking','Short dwell','Blocking','3C Experiment']

for i in range(4):
    contact_map_file = contact_map_files[i]
    output_label="contact_law_combined"
    locus_size=1000
    mat = np.load(contact_map_file)
    N = mat.shape[0]
    # Align and average diagonals
    aligned_mat = contact_mat_alignment(mat)
    contact_sum = np.mean(aligned_mat, axis=0)
    ax.plot(za, contact_sum[lower_lim:upper_lim],
            c=colors[i], lw=1.0, alpha=0.35, marker='.', markersize=6,
            fillstyle='none', label=labels[i])

    #multi_fits(ax, locus_size, regimes, contact_sum)

ax.set_xlim(za[0], za[-1])
ax.set_ylim(.00005, 1)
ax.set_yscale('log')
ax.set_xscale('log')
ax.legend(fontsize=12, loc='upper right')
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)
plt.tight_layout()

# Create twin y-axis
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())  # match y-limits
ax2.set_yscale('log')        # match log scale
ax2.tick_params(axis='y', labelleft=False, labelright=False, right=True, left=False)

out_pdf = f"{output_label}_contact_law.pdf"
out_png = f"{output_label}_contact_law.png"
plt.savefig(out_pdf, dpi=300)
plt.savefig('./figures/contact_law.png', dpi=300)
print(f"✅ Saved plots: {out_pdf}, {out_png}")

In [ ]:
import numpy as np
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import matplotlib.cm as colormaps
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import FuncFormatter
mm = 1/25.4

CG = 100
CG *= 10

def recenter_contact_matrix(contact_matrix, origin=None):
    N = contact_matrix.shape[0]
    if origin is None:
        origin = N // 2

    # roll rows and columns
    rolled = np.roll(contact_matrix, -origin, axis=0)
    rolled = np.roll(rolled, -origin, axis=1)

    return rolled

def plot_contact_matrix(date, file, vmin=0.0002, vmax = 0.1, choose=False):


    contact_matrix_normalized = np.load(file)

    fig_size = [65*2,55*2]

    fig = plt.figure(figsize=(fig_size[0]*mm,fig_size[1]*mm))

    ax = plt.gca()

    #how to choose vmax? ben default is 0.025
    # vmax = .002
    # map_norm = colors.Normalize(vmin=0.0, vmax=vmax)
#     map_norm = colors.LogNorm(vmin=np.min(contact_matrix_normalized[contact_matrix_normalized != 0]), vmax=np.max(contact_matrix_normalized))

#     # cmap = 'Reds'
#     cmap = 'plasma'
#     im = ax.imshow(contact_matrix_normalized,cmap=cmap,norm=map_norm)

    masked_matrix = np.ma.masked_equal(contact_matrix_normalized, 0)

    # Copy colormap and set masked (zero) entries to black
    # cmap = plt.cm.plasma.copy()
    # cmap.set_bad(color='black')
    # cmap = colors.LinearSegmentedColormap.from_list(
    # 'white_purple_black', ['white', 'lightblue', 'purple',
    #                        'purple', 'purple', 'black'])

    # Log normalization, ignoring zeros
    if not choose:
        map_norm = colors.LogNorm(
            vmin=np.min(contact_matrix_normalized[contact_matrix_normalized != 0]),
            vmax=np.max(contact_matrix_normalized)
        )
        vmin=np.min(contact_matrix_normalized[contact_matrix_normalized != 0]),
        vmax=np.max(contact_matrix_normalized)
        print(vmin,vmax)
    else:
        map_norm = colors.LogNorm(
            vmin=vmin,
            vmax=vmax
        )

    # Display
    im = ax.imshow(masked_matrix, cmap=cmap, norm=map_norm, interpolation='nearest')

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)

    cbar = fig.colorbar(im, cax=cax)
    
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label(label=r'Contact Probability', fontsize=12)
    cbar.ax.tick_params(labelsize=12)

    ax.set_xlabel(r'Genomic Position [kbp]', fontsize=12)
    ax.set_ylabel(r'Genomic Position [kbp]', fontsize=12)

    scale = 1
    formatter = FuncFormatter(lambda x, _: "{:.0f}".format(x*scale))

    ax.xaxis.set_major_formatter(formatter)
    ax.yaxis.set_major_formatter(formatter)

    ax.tick_params(axis='x', labelsize=12)
    ax.tick_params(axis='y', labelsize=12)

    plt.tight_layout()

    fig.savefig('./figures/'+ date +".png", dpi=300)
    print("Saved contact map figure for given range")
    plt.show()

In [ ]:
cmap = colors.LinearSegmentedColormap.from_list(
    'white_purple_black', ['white', 
                           'red', 'black'])
cmap = 'Reds'

In [ ]:
plot_contact_matrix('Nov18', "Nov18.npy", vmin=.001, vmax=.1, choose=False)

In [ ]:
plot_contact_matrix('Oct30', "Oct30.npy", vmin=.001, vmax=.1, choose=False)

In [ ]:
plot_contact_matrix('Oct30', "Oct30.npy", vmin=.001, vmax=.1, choose=True)

In [ ]:
plot_contact_matrix('Oct29', 'Oct29.npy', vmin=.0004, vmax=.1, choose=False)

In [ ]:
plot_contact_matrix('Oct29', 'Oct29.npy', vmin=.0006, vmax=.1, choose=True)

In [ ]:
plot_contact_matrix('Nov04', 'Nov04.npy', vmin=.001, vmax=.1, choose=False)

In [ ]:
plot_contact_matrix('Nov04', 'Nov04.npy', vmin=.001, vmax=.1, choose=True)

In [ ]:
plot_contact_matrix('3C_contact_matrix_normalized', '3C_contact_matrix_normalized.npy',vmin=.0001, vmax=1, choose=True)

In [ ]:
plot_contact_matrix('3C_contact_matrix_normalized', '3C_contact_matrix_normalized.npy', choose=False)

In [ ]:
def read_bin(path, order="row"):
    data = np.fromfile(path, dtype=np.float64)
    n_coords = data.size
    if n_coords % 3 != 0:
        raise ValueError(f"{path} size not divisible by 3")
    N = n_coords // 3
    if order == "row":
        coords = data.reshape((N, 3))
    elif order == "col":
        coords = np.empty((N, 3))
        coords[:, 0] = data[0:N]
        coords[:, 1] = data[N:2*N]
        coords[:, 2] = data[2*N:3*N]
    else:
        raise ValueError("order must be 'row' or 'col'")
    return coords

def voxel_contact_map(coords, seg_ids, n_segments, cutoff, box_length=None):
    """
    Compute segment–segment contact map using voxel neighbor search.

    coords : (N,3) bead coordinates
    seg_ids : (N,) array, segment index for each bead
    n_segments : int, total number of segments
    cutoff : float, contact distance
    box_length : optional float, for cubic periodic box
    """
    N = len(coords)
    cell_size = cutoff
    min_coord = coords.min(axis=0)
    max_coord = coords.max(axis=0)
    box_extent = (max_coord - min_coord).max()
    ncell = int(np.ceil(box_extent / cell_size))

    # map beads to voxels
    cell_idx = np.floor((coords - min_coord) / cell_size).astype(int) % ncell
    flat_idx = cell_idx[:,0] + ncell*(cell_idx[:,1] + ncell*cell_idx[:,2])

    cell_particles = [[] for _ in range(ncell**3)]
    for i, c in enumerate(flat_idx):
        cell_particles[c].append(i)

    contacts = np.zeros((n_segments, n_segments), dtype=int)

    # neighbor offsets (27 total)
    offsets = [(dx,dy,dz) for dx in (-1,0,1)
                          for dy in (-1,0,1)
                          for dz in (-1,0,1)]

    for cx in range(ncell):
        for cy in range(ncell):
            for cz in range(ncell):
                cflat = cx + ncell*(cy + ncell*cz)
                beads1 = cell_particles[cflat]
                if not beads1:
                    continue
                for dx,dy,dz in offsets:
                    nx, ny, nz = (cx+dx)%ncell, (cy+dy)%ncell, (cz+dz)%ncell
                    nflat = nx + ncell*(ny + ncell*nz)
                    beads2 = cell_particles[nflat]
                    for i in beads1:
                        for j in beads2:
                            if j <= i:  # avoid double count
                                continue
                            d = coords[i] - coords[j]
                            if box_length is not None:
                                d -= box_length*np.round(d/box_length)
                            if np.dot(d,d) < cutoff**2:
                                si, sj = seg_ids[i], seg_ids[j]
                                # if si != sj:
                                contacts[si, sj] = 1
                                contacts[sj, si] = 1
    return contacts

In [ ]:
from pathlib import Path
import sys
base_dir = Path("/projects/bdxm/amaytin/")  # adjust for server
# tag='Sep28'
segsize = 100
import numpy as np

def calculate_contact_matrix(tag, minute):
    reps = {
        tag: [1,2,3,4,5,6,7,8,9,10],
    }
    total_reads = 0
    N = 54338*2
    seg_ids = np.arange(N) // segsize
    n_segments = seg_ids.max() + 1
    N_CG = seg_ids.max() + 1
    contact_sum = np.zeros((N_CG,N_CG), dtype=np.int32)
    # minute=90

    for date, rep_list in reps.items():
        for rep in rep_list:
            # First check if the grouped folder exists
            grouped_path = base_dir / date / f"{date}_{rep}" / "data" / "coords"
            standalone_path = base_dir / f"{date}_{rep}" / "data" / "coords"

            if grouped_path.exists():
                rep_dir = grouped_path
            elif standalone_path.exists():
                rep_dir = standalone_path
            else:
                raise FileNotFoundError(f"No DNA folder found for {date}_{rep}")

            # Gather bin files
            bin_files = list(rep_dir.glob(f"dna_{date}_{rep}_{minute}.bin"))
            total_files = len(bin_files)

            #coords = read_bin(bin_files[0])
            #N = len(coords)
            #seg_ids = np.arange(N) // segsize
            #n_segments = seg_ids.max() + 1
            #contacts = np.zeros((n_segments, n_segments), dtype=np.int32)

            for i, bin_file in enumerate(bin_files, start=1):
                coords = read_bin(bin_file)
                ival = voxel_contact_map(coords, seg_ids, n_segments, cutoff=60.0)
                contact_sum += ival
                total_reads += np.count_nonzero(ival)

                if i % 10 == 0 or i == total_files:
                    print("Contact matrix shape:", contact_sum.shape)
                    print("Thresholded nonzeros:", np.count_nonzero(contact_sum))
                    print("Total number of counts (reads):", total_reads)
                    print("Min/max/mean:", contact_sum.min(), contact_sum.max(), contact_sum.mean())
                    pct = (i / total_files) * 100
                    print(f"  {i}/{total_files} ({pct:.1f}%) done for {date}_{rep}")

    # Save aggregated contact map
    np.save("./results/"+date+"/contact_map_delta_cg"+str(segsize*10)+tag+"_"+minute+".npy", contact_sum)
    print("Final contact map saved as contact_map_delta_cg"+str(segsize*10)+tag+"_"+minute+".npy")

In [ ]:
calculate_contact_matrix('Nov21','3')

In [ ]:
calculate_contact_matrix('Nov21','17')

In [ ]:
calculate_contact_matrix('Oct22','46')

In [ ]:
CG = 100
CG *= 10
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import matplotlib.cm as colormaps
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import FuncFormatter
import numpy as np
mm = 1/25.4

def plot_contact_matrix_full(date, files, vmin=0.0002, vmax = 0.1, gmax=543, choose=False):

    contact_matrix_normalized = np.zeros_like(np.load(files[0]))
    for file in files:
        contact_matrix_normalized += np.load(file)
    
    if np.all(np.diag(contact_matrix_normalized) > 0):
        print("Diagonal is all 10.")
    else:
        print("Diagonal is NOT all 10.")

    fig_size = [65*2,55*2]

    fig = plt.figure(figsize=(fig_size[0]*mm,fig_size[1]*mm))

    ax = plt.gca()

    #how to choose vmax? ben default is 0.025
    # vmax = .002
    # map_norm = colors.Normalize(vmin=0.0, vmax=vmax)
#     map_norm = colors.LogNorm(vmin=np.min(contact_matrix_normalized[contact_matrix_normalized != 0]), vmax=np.max(contact_matrix_normalized))

#     # cmap = 'Reds'
#     cmap = 'plasma'
#     im = ax.imshow(contact_matrix_normalized,cmap=cmap,norm=map_norm)

    masked_matrix = np.ma.masked_equal(contact_matrix_normalized, 0)

    # Copy colormap and set masked (zero) entries to black
    # cmap = plt.cm.plasma.copy()
    # cmap.set_bad(color='black')
    # cmap = colors.LinearSegmentedColormap.from_list(
    # 'white_purple_black', ['white', 'purple', 'black'])


    # Log normalization, ignoring zeros
    # if not choose:
    #     map_norm = colors.LogNorm(
    #         vmin=np.min(contact_matrix_normalized[contact_matrix_normalized != 0]),
    #         vmax=np.max(contact_matrix_normalized)
    #     )
    #     vmin=np.min(contact_matrix_normalized[contact_matrix_normalized != 0]),
    #     vmax=np.max(contact_matrix_normalized)
    #     print(vmin,vmax)
    # else:
    #     map_norm = colors.LogNorm(
    #         vmin=vmin,
    #         vmax=vmax
    #     )
    
    #vmin=np.min(contact_matrix_normalized[contact_matrix_normalized != 0]),
    #vmax=np.max(contact_matrix_normalized)
        
    map_norm = colors.Normalize(vmin=vmin, vmax=vmax)

    # Display
    im = ax.imshow(masked_matrix, cmap=cmap, norm=map_norm, interpolation='nearest')
    ax.set_xlim(0, gmax)
    ax.set_ylim(gmax, 0)
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)

    cbar = fig.colorbar(im, cax=cax)
    
    cbar = fig.colorbar(im, cax=cax, ticks=[1,2,3,4])
    cbar.set_label(label=r'Contact Count', fontsize=12)
    cbar.ax.tick_params(labelsize=12)

    ax.set_xlabel(r'Genomic Position [kbp]', fontsize=12)
    ax.set_ylabel(r'Genomic Position [kbp]', fontsize=12)

    scale = 1
    formatter = FuncFormatter(lambda x, _: "{:.0f}".format(x*scale))

    ax.xaxis.set_major_formatter(formatter)
    ax.yaxis.set_major_formatter(formatter)

    ax.tick_params(axis='x', labelsize=12)
    ax.tick_params(axis='y', labelsize=12)

    plt.tight_layout()

    fig.savefig("./figures/"+date+"_full.png", dpi=300)
    print("Saved contact map figure")
    plt.show()

In [ ]:
cmap = colors.LinearSegmentedColormap.from_list(
    'white_purple_black', ['white', 'purple', 'black'])

cmap = 'Reds'

In [ ]:
plot_contact_matrix_full('Oct22_3',
                         ['./results/Oct22/contact_map_delta_cg1000Oct22_2.npy',
                         './results/Oct22/contact_map_delta_cg1000Oct22_3.npy',
                         './results/Oct22/contact_map_delta_cg1000Oct22_4.npy'],
                         vmax=3,
                        gmax=590)

In [ ]:
plot_contact_matrix_full('Oct22_3',
                         ['./results/Oct22/contact_map_delta_cg1000Oct22_3.npy'],
                         vmax=3,
                        gmax=590)

In [ ]:
plot_contact_matrix_full('Oct22_17',
                         ['./results/Oct22/contact_map_delta_cg1000Oct22_16.npy',
                          './results/Oct22/contact_map_delta_cg1000Oct22_17.npy',
                          './results/Oct22/contact_map_delta_cg1000Oct22_18.npy'],
                         vmax=3,
                        gmax=760)


In [ ]:
plot_contact_matrix_full('Oct22_46',
                         ['./results/Oct22/contact_map_delta_cg1000Oct22_45.npy',
                          './results/Oct22/contact_map_delta_cg1000Oct22_46.npy',
                          './results/Oct22/contact_map_delta_cg1000Oct22_47.npy'],
                         vmax=3,
                        gmax=1086)

In [ ]:
import numpy as np

M = np.load('./results/Oct22/contact_map_delta_cg1000Oct22_60.npy')  # NxN numpy array
d1 = np.diag(M, k=1)  # +1 diagonal
zero_idx = np.where(d1 == 0)[0]

print("First 20 zero idx:", zero_idx[:20])
print("Diffs:", np.diff(zero_idx)[:20])
print("Unique diffs:", np.unique(np.diff(zero_idx)))

In [ ]:
plot_contact_matrix_full('Nov21_3',
                         ['./results/Nov21/contact_map_delta_cg1000Nov21_3.npy'],
                         vmax=3,
                        gmax=590)